In [1]:
pip install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 135.0 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [2]:
from ultralytics import YOLO
import itertools
import pandas as pd
import os
import torch
from roboflow import Roboflow
import zipfile
import shutil
import os
from google.colab import files

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
rf = Roboflow(api_key="a1uozLDJeFIwfWwaSoy6")
project = rf.workspace("s-gbust").project("armed-person-recognition-gohff")
version = project.version(4)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Armed-Person-Recognition-4 in yolov8:: 100%|██████████| 17366/17366 [00:02<00:00, 8349.44it/s]


In [4]:
# === Grid Parameter ===
optimizers = ["AdamW"]
learning_rates = [0.001, 0.01]
weight_decays = [0.0005]

# Buat kombinasi parameter
param_combinations = list(itertools.product(optimizers, learning_rates, weight_decays))
results = []

# Loop semua kombinasi parameter
for i, (optimizer, lr, wd) in enumerate(param_combinations):
    exp_name = f"exp_{i}_{optimizer}_lr{lr}_wd{wd}"
    print(f"\n🚀 === Training {exp_name} ===")

    try:
        # Load model YOLO
        model = YOLO("yolov8s.pt")  # ganti sesuai kebutuhan

        # Jalankan training
        model.train(
            data="/content/Armed-Person-Recognition-4/data.yaml",
            epochs=50,
            lr0=lr,
            weight_decay=wd,
            batch=64,
            imgsz=640,
            project="gridsearch_results_YOLOv8s_part1",
            name=exp_name,
            optimizer=optimizer,
            exist_ok=True,
            amp=False,
            deterministic=True,
            patience=30
        )

        # Ambil hasil training
        results_path = os.path.join("gridsearch_results", exp_name, "results.csv")
        if os.path.exists(results_path):
            df = pd.read_csv(results_path)
            last_row = df.iloc[-1]
            mAP50 = model.val(data="/content/Armed-Person-Recognition-4/data.yaml", task='test', save=True)
            mAP50 = mAP50.box.map50

            if pd.notna(mAP50):
                print(f"✅ {exp_name} → mAP50: {mAP50:.4f}")
                results.append({
                    "exp_name": exp_name,
                    "optimizer": optimizer,
                    "learning_rate": lr,
                    "weight_decay": wd,
                    "mAP50": mAP50
                })
            else:
                print(f"⚠️ {exp_name} → mAP50 not found (check training logs).")

        else:
            print(f"⚠️ Results file not found for {exp_name}")

    except Exception as e:
        print(f"❌ Error saat training {exp_name}: {e}")
        continue

# Simpan hasil
if results:
    results_df = pd.DataFrame(results)
    results_df.sort_values(by="mAP50", ascending=False, inplace=True)
    results_df.to_csv("yolo_gridsearch_results.csv", index=False)
    print("\n=== 🏁 Grid Search Complete! ===")
    print(results_df)
else:
    print("\n⚠️ Tidak ada hasil valid. Periksa error log di atas.")


🚀 === Training exp_0_AdamW_lr0.001_wd0.0005 ===
Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Armed-Person-Recognition-4/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_0_AdamW_lr0.001_wd0.0005, nbs=64, nms=False, opset=Non

In [5]:
folder_path = "/content/gridsearch_results_YOLOv8s_part1" # This is the folder name from the output of the training run
zip_path = "/content/gridsearch_results_YOLOv8s_part1.zip"

if os.path.exists(folder_path):
    shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)
    print(f"Folder '{folder_path}' zipped to '{zip_path}'")
    files.download(zip_path)
else:
    print(f"Folder '{folder_path}' not found.")

Folder '/content/gridsearch_results_YOLOv8s_part1' zipped to '/content/gridsearch_results_YOLOv8s_part1.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# === Grid Parameter ===
optimizers = ["SGD"]
learning_rates = [0.001, 0.01]
weight_decays = [0.0005]

# Buat kombinasi parameter
param_combinations = list(itertools.product(optimizers, learning_rates, weight_decays))
results = []

# Loop semua kombinasi parameter
for i, (optimizer, lr, wd) in enumerate(param_combinations):
    exp_name = f"exp_{i}_{optimizer}_lr{lr}_wd{wd}"
    print(f"\n🚀 === Training {exp_name} ===")

    try:
        # Load model YOLO
        model = YOLO("yolov8s.pt")  # ganti sesuai kebutuhan

        # Jalankan training
        model.train(
            data="/content/Armed-Person-Recognition-4/data.yaml",
            epochs=50,
            lr0=lr,
            weight_decay=wd,
            batch=64,
            imgsz=640,
            project="gridsearch_results_YOLOv8s_part2",
            name=exp_name,
            optimizer=optimizer,
            exist_ok=True,
            amp=False,
            deterministic=True,
            patience=30
        )

        # Ambil hasil training
        results_path = os.path.join("gridsearch_results", exp_name, "results.csv")
        if os.path.exists(results_path):
            df = pd.read_csv(results_path)
            last_row = df.iloc[-1]
            mAP50 = model.val(data="/content/Armed-Person-Recognition-4/data.yaml", task='test', save=True)
            mAP50 = mAP50.box.map50

            if pd.notna(mAP50):
                print(f"✅ {exp_name} → mAP50: {mAP50:.4f}")
                results.append({
                    "exp_name": exp_name,
                    "optimizer": optimizer,
                    "learning_rate": lr,
                    "weight_decay": wd,
                    "mAP50": mAP50
                })
            else:
                print(f"⚠️ {exp_name} → mAP50 not found (check training logs).")

        else:
            print(f"⚠️ Results file not found for {exp_name}")

    except Exception as e:
        print(f"❌ Error saat training {exp_name}: {e}")
        continue

# Simpan hasil
if results:
    results_df = pd.DataFrame(results)
    results_df.sort_values(by="mAP50", ascending=False, inplace=True)
    results_df.to_csv("yolo_gridsearch_results.csv", index=False)
    print("\n=== 🏁 Grid Search Complete! ===")
    print(results_df)
else:
    print("\n⚠️ Tidak ada hasil valid. Periksa error log di atas.")


🚀 === Training exp_0_SGD_lr0.001_wd0.0005 ===
Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Armed-Person-Recognition-4/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_0_SGD_lr0.001_wd0.0005, nbs=64, nms=False, opset=None, o

In [7]:
folder_path = "/content/gridsearch_results_YOLOv8s_part2" # This is the folder name from the output of the training run
zip_path = "/content/gridsearch_results_YOLOv8s_part2.zip"

if os.path.exists(folder_path):
    shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)
    print(f"Folder '{folder_path}' zipped to '{zip_path}'")
    files.download(zip_path)
else:
    print(f"Folder '{folder_path}' not found.")

Folder '/content/gridsearch_results_YOLOv8s_part2' zipped to '/content/gridsearch_results_YOLOv8s_part2.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>